### Car Failure Prediction: Business-Focused EDA and Classification Target Design

#### Executive Summary
This notebook analyzes operational and categorical vehicle data to quantify failure risk patterns and create a machine-learning-ready Fail or Pass target. The analysis is structured to support preventive maintenance decisions by showing not only what is happening in the data, but also why the patterns are operationally meaningful.

#### Business Objective
The core objective is to identify the combinations of vehicle segment and operating conditions most associated with higher failure risk. These findings are intended to guide earlier intervention, reduce unplanned downtime, and improve maintenance resource allocation.

#### Scope of this Notebook
- Load and profile data from Failure.csv
- Assess data quality and invalid readings
- Standardize schema and feature naming
- Apply targeted preprocessing for RPM, temperature, fuel consumption, and membership
- Explore feature-level failure patterns with interpretation
- Define a binary Fail or Pass target for classification
- Summarize findings, limitations, and production-oriented next steps

#### Competencies Demonstrated
- SQL-first analytics with DuckDB in Python
- Practical data quality treatment with transparent assumptions
- Feature engineering for interpretability and decision support
- Translation of analytical outputs into business implications

In [ ]:
import pandas as pd
import duckdb

#### Data Ingestion

The dataset is loaded into DuckDB as a working analytical table. This setup enables reproducible SQL-based profiling and makes each cleaning or transformation step auditable.

In [3]:
import duckdb

con = duckdb.connect("failure.db")

# 2. Import the CSV file into a real DuckDB table
con.execute("""
CREATE OR REPLACE TABLE failure AS
SELECT *
FROM read_csv_auto('Failure.csv')
""")


#### Initial Data Preview

The first full-table preview is used to validate schema structure, spot obvious anomalies, and establish a baseline before any preprocessing. This step helps separate data issues from true business signals later in the analysis.

In [4]:
query = """
select *
from failure

""" 

con.execute(query).df()

,Car ID,Model,Color,Temperature,RPM,Factory,Usage,Fuel consumption,Membership,Failure A,Failure B,Failure C,Failure D,Failure E
0,CAR-00001,Pickup,Yellow,92.0,4846.0,Factory C,High,9.41,None,0,1,0,0,0
1,CAR-00002,Sedan,Yellow,100.4,5057.0,Factory D,Very High,9.54,Basic,0,0,0,0,0
2,CAR-00003,Pickup,Green,64.9,2770.0,Factory C,Low,NaN,None,0,0,0,0,0
3,CAR-00004,Hatchback,Gray,76.0,2749.0,Factory D,Medium,8.07,Gold,0,1,0,0,0
4,CAR-00005,SUV,Blue,78.2,4755.0,Factory A,Medium,9.45,None,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10076,CAR-10077,Wagon,Blue,78.4,2697.0,Factory C,Low,6.91,Platinum,0,0,0,0,0
10077,CAR-10078,Van,Red,79.2,2554.0,Factory B,Low,7.64,Silver,0,0,0,0,0
10078,CAR-10079,Hatchback,Blue,77.0,4292.0,Factory A,High,8.45,Gold,0,0,0,0,0
10079,CAR-10080,Hatchback,Red,92.9,4349.0,Factory A,High,9.94,Gold,0,0,0,1,0


#### Data Quality Assessment

This check quantifies missing and non-physical values across core fields. The objective is to identify where raw data quality could bias failure-rate comparisons if left untreated.

In [5]:
query = """
select sum(case when "Temperature" is null or "Temperature" <= 0 then 1 else 0 end) as bad_temp,
sum(case when "RPM" is null or "RPM" <= 0 then 1 else 0 end) as bad_rpm,
sum(case when "Fuel consumption" is null or "Fuel consumption" <= 0 then 1 else 0 end) as bad_fuel_consumption,
sum(case when "Membership" is null then 1 else 0 end) as bad_membership,
sum(case when "color" is null then 1 else 0 end) as bad_color,
sum(case when "Factory" is null then 1 else 0 end) as bad_factory,
sum(case when "Usage" is null then 1 else 0 end) as bad_usage,
sum(case when "Model" is null then 1 else 0 end) as bad_model
from failure

""" 

con.execute(query).df()

,bad_temp,bad_rpm,bad_fuel_consumption,bad_membership,bad_color,bad_factory,bad_usage,bad_model
0,213.0,146.0,236.0,2032.0,0.0,0.0,0.0,0.0


#### Schema Standardization

Columns are normalized to lowercase naming so SQL transformations remain consistent and less error-prone. This is a maintainability step that reduces downstream ambiguity in joins, filters, and feature definitions.

In [6]:
con.execute('alter table failure rename column "Car ID" to car_id')
con.execute('alter table failure rename column "Model" to model')
con.execute('alter table failure rename column "Color" to color')
con.execute('alter table failure rename column "RPM" to rpm')
con.execute('alter table failure rename column "Factory" to factory')
con.execute('alter table failure rename column "Usage" to usage')
con.execute('alter table failure rename column "Fuel consumption" to fuel_consumption')
con.execute('alter table failure rename column "Membership" to membership')
con.execute('alter table failure rename column "Failure A" to failure_a')
con.execute('alter table failure rename column "Failure B" to failure_b')
con.execute('alter table failure rename column "Failure C" to failure_c')
con.execute('alter table failure rename column "Failure D" to failure_d')
con.execute('alter table failure rename column "Failure E" to failure_e')
con.execute('alter table failure rename column "Temperature" to temp')

#### Numeric Value Treatment

Invalid or missing numeric readings are imputed using model-level medians. This preserves model-specific operating behavior while limiting distortion from outliers and impossible values such as non-positive RPM or temperature.

In [7]:
con.execute("""
update failure as f 
set rpm = m.median_rpm 
from (select model, median(rpm) as median_rpm 
from failure 
where "rpm" > 0 
group by model) as m 
where f.model = m.model and (f.rpm <= 0 or f.rpm is null)
""")

con.execute(""" 
update failure as f 
set fuel_consumption = median_fuel 
from (select model, median(fuel_consumption) as median_fuel 
from failure 
where "fuel_consumption" > 0 
group by model) as fc 
where f.model = fc.model and (f.fuel_consumption <= 0 or f.fuel_consumption is null) 
""")

con.execute("""
update failure as f
set temp = median_temp
from (select model, median(temp) as median_temp
from failure
where "temp" > 0
group by model) as m
where f.model = m.model and (f.temp is null or f.temp <= 0)   
""")



#### Missingness Pattern: Membership Variable

The missingness profile is concentrated in membership, indicating the issue is not broadly distributed across features. Treating missing membership as No Membership preserves records and captures the potential signal that non-enrollment itself may relate to failure behavior.

#### Missing Values Overview

This step verifies where null patterns remain prior to modeling logic. The analysis focuses on whether missingness is systematic enough to influence class-level comparisons and whether additional treatment is required.

In [8]:
con.execute("""
update failure
set membership = 'No Membership'
where membership is null or membership = 'None'
""")

#### Membership Standardization

Null and None labels are consolidated into No Membership to avoid fragmented category counts. This improves interpretability when comparing membership segments against failure outcomes.

In [9]:
query = """
select membership, count(*) as no_membership
from failure
group by membership

""" 

con.execute(query).df()

,membership,no_membership
0,Basic,2075
1,Gold,2017
2,Platinum,1990
3,Silver,1967
4,No Membership,2032


#### Post-Cleaning Quality Check

After preprocessing, this validation confirms that critical fields are in a usable state for target engineering and cohort-level analysis. Any residual nulls here would indicate remaining data preparation risk.

In [10]:
query = """
select sum(case when "car_id" is null then 1 else 0 end) as car_id_null, 
sum(case when "model" is null then  1 else 0 end) as model_null,
sum(case when "color" is null then 1 else 0 end) as color_null,
sum(case when "temp" is null then 1 else 0 end) as temperature_null,
sum(case when "rpm" is null then 1  else 0 end) as rpm_null,
sum(case when "factory" is null then 1 else 0 end) as factory_null,
sum(case when "usage" is null then 1 else 0 end) as usage_null,
sum(case when "fuel_consumption" is null then 1 else 0 end) as fuel_consumption_null,
sum(case when "membership" is null then 1 else 0 end) as membership_null,
sum(case when "failure_a" is null then 1 else 0 end) as failure_a_null,
sum(case when "failure_b" is null then 1 else 0 end) as failure_b_null,
sum(case when "failure_c" is null then 1 else 0 end) as failure_c_null,
sum(case when "failure_d" is null then 1 else 0 end) as failure_d_null,
sum(case when "failure_e" is null then 1 else 0 end) as failure_e_null
from failure
"""

con.execute(query).df()

,car_id_null,model_null,color_null,temperature_null,rpm_null,factory_null,usage_null,fuel_consumption_null,membership_null,failure_a_null,failure_b_null,failure_c_null,failure_d_null,failure_e_null
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Feature Grouping Strategy

Features are separated into target and predictors to keep analysis logic explicit.

- Target variables: failure_a through failure_e
- Predictor variables: all remaining operational and categorical fields

This separation clarifies which variables explain risk and which variable represents the prediction objective.

#### Binary Target Definition

The five failure indicators are consolidated into one classification label to make model training and business communication more practical.

- Fail: at least one failure indicator equals 1
- Pass: all failure indicators equal 0

This transformation shifts the problem from multi-flag monitoring to an actionable pass or fail decision framework.

In [11]:
query = """
select case when coalesce(failure_a, 0) + coalesce(failure_b, 0) + coalesce(failure_c, 0) + coalesce(failure_d, 0) + coalesce(failure_e, 0) > 0 then 'fail' else 'pass' end as car_status, count(*) as count, 
round(100.0 * count(*) / sum(count(*)) over(), 2) as pct
from failure
group by car_status

"""

con.execute(query).df()

,car_status,count,pct
0,fail,3614,35.85
1,pass,6467,64.15


#### Model-Level Failure Pattern

The model-level comparison indicates SUV has the highest observed failure rate (38.24%), while Hatchback has the lowest (32.49%). The spread suggests model type contributes measurable predictive signal, though differences should be interpreted alongside usage and operating-condition effects.

In [12]:
query = """
with total_failure_rate as (
select model as models, count(*) as total_cars
from failure
group by model),
failure_rate as (
select model as failed_models, count(*) as failed_cars
from failure 
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by model)
select *, round(100.0 * failed_cars / total_cars, 2) as failure_rate_pct
from failure_rate as fr
left join total_failure_rate as tfr on fr.failed_models = tfr.models
order by failure_rate_pct desc


"""

con.execute(query).df()

,failed_models,failed_cars,models,total_cars,failure_rate_pct
0,SUV,574,SUV,1501,38.24
1,Coupe,508,Coupe,1355,37.49
2,Pickup,502,Pickup,1366,36.75
3,Van,555,Van,1513,36.68
4,Wagon,518,Wagon,1431,36.20
5,Sedan,481,Sedan,1450,33.17
6,Hatchback,476,Hatchback,1465,32.49


#### Factory-Level Failure Pattern

Factory-level failure rates are not uniform: Factory B is highest and Factory A is lowest. This pattern suggests potential manufacturing-process or quality-control variation that may be useful for risk stratification.

In [13]:
query = """
with failure_rate_factory as (select factory as failed_factory, count(*) as no_failed_factory
from failure
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by factory),
total_factory as (select factory, count(*) as total_factory
from failure
group by factory)
select *, round(100.0 * no_failed_factory / total_factory, 2) as failed_factory_pct
from failure_rate_factory as frf 
left join total_factory as tf on frf.failed_factory = tf.factory
order by failed_factory_pct desc
"""

con.execute(query).df()

,failed_factory,no_failed_factory,factory,total_factory,failed_factory_pct
0,Factory B,739,Factory B,1937,38.15
1,Factory D,772,Factory D,2083,37.06
2,Factory E,723,Factory E,2067,34.98
3,Factory C,694,Factory C,1999,34.72
4,Factory A,686,Factory A,1995,34.39


#### Usage and Failure Relationship

Failure rates rise materially with higher usage bands, indicating operational intensity is a strong risk correlate. This reinforces usage as a high-value predictor and a practical maintenance-priority signal.

In [14]:
query = """
with total_usage as (select usage, count(*) as total_no_usage
from failure
group by usage),
usage_group as (select usage as failed_usage, count(*) as no_failed_usage
from failure
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by usage 
)
select *, round(100.0 * no_failed_usage / total_no_usage, 2) as failed_usage_pct
from usage_group as ug
left join total_usage as tu on ug.failed_usage = tu.usage
order by failed_usage_pct desc


"""

con.execute(query).df()

,failed_usage,no_failed_usage,usage,total_no_usage,failed_usage_pct
0,Very High,700,Very High,981,71.36
1,High,1213,High,2503,48.46
2,Medium,1236,Medium,4011,30.82
3,Low,465,Low,2586,17.98


#### Model-Level Pass or Fail Distribution

This breakdown shows how fail versus pass volume is distributed across model categories. It helps determine whether risk concentration is driven by rate differences, segment size, or both.

In [15]:
query = """
select model, case when failure_a + failure_b + failure_c + failure_d + failure_e > 0 then 'fail' else 'pass' end as car_status, count(*) as count, round(100.0 * count(*) / sum(count(*)) over(), 2) as pct
from failure
group by model, car_status
order by count desc
"""

con.execute(query).df()

,model,car_status,count,pct
0,Hatchback,pass,989,9.81
1,Sedan,pass,969,9.61
2,Van,pass,958,9.50
3,SUV,pass,927,9.20
4,Wagon,pass,913,9.06
5,Pickup,pass,864,8.57
6,Coupe,pass,847,8.40
7,SUV,fail,574,5.69
8,Van,fail,555,5.51
9,Wagon,fail,518,5.14


#### Feature Engineering: Risk Bands

Continuous operating variables are converted into risk bands to improve interpretability and segment-level reasoning.

- RPM: Low, Medium, High, Danger
- Temperature: Low, Medium, High
- Fuel consumption: Low, Medium, High

These grouped features make it easier to communicate where failure risk accelerates and to test nonlinear effects in downstream models.

In [16]:
query = """
select *, case when rpm between 0 and 1500 then 'Low'
when rpm between 1501 and 3000 then 'Medium'
when rpm between 3001 and 4500 then 'High'
when rpm > 4500 then 'Danger' end as rpm_class,

case when temp < 75 then 'Low'
when temp >= 75 and temp < 105 then 'Medium'
when temp >= 105 then 'High' end as temp_class,

case when fuel_consumption < 5 then 'Low'
when fuel_consumption >= 5 and fuel_consumption <= 8 then 'Medium' 
when fuel_consumption > 8 then 'High' end as fuel_class
from failure
 
"""

failure_fe = con.execute(query).df()
con.register('failure_fe', failure_fe)

#### RPM Class Insight

The Danger RPM band has the highest failure rate, suggesting failure risk increases under extreme engine-speed conditions. This supports RPM class as an interpretable proxy for mechanical stress.

In [17]:
query = """
with total_rpm as (select rpm_class as total_rpm_class, count(*) as total_no_rpm
from failure_fe
group by rpm_class),
failed_rpm as (select rpm_class, count(*) as no_failed_rpm
from failure_fe
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by rpm_class)
select *, round(100.0 * no_failed_rpm / total_no_rpm, 2) as failed_rpm_pct
from failed_rpm as fr
left join total_rpm as tr on fr.rpm_class = tr.total_rpm_class 
order by failed_rpm_pct desc
"""

con.execute(query).df()

,rpm_class,no_failed_rpm,total_rpm_class,total_no_rpm,failed_rpm_pct
0,Danger,628,Danger,973,64.54
1,High,2011,High,4941,40.70
2,Medium,933,Medium,3880,24.05
3,Low,42,Low,287,14.63


#### Membership Segment Insight

Platinum membership appears with the highest observed failure rate. This likely reflects underlying behavioral differences, such as usage intensity, rather than membership tier alone; therefore, interpretation should remain multivariate.

In [24]:
query = """
with all_membership as (select membership, count(*) as no_total_membership
from failure_fe
group by membership),
failed_membership as (select membership as fail_membership, count(*) as no_failed_membership
from failure_fe
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by fail_membership)
select *, round(100.0 * no_failed_membership / no_total_membership, 2) as membership_pct
from failed_membership as fm
left join all_membership as am on fm.fail_membership = am.membership
order by membership_pct desc

"""

con.execute(query).df()

,fail_membership,no_failed_membership,membership,no_total_membership,membership_pct
0,Platinum,721,Platinum,1990,36.23
1,No Membership,734,No Membership,2032,36.12
2,Silver,709,Silver,1967,36.04
3,Gold,716,Gold,2017,35.50
4,Basic,734,Basic,2075,35.37


#### Fuel Consumption Insight

Vehicles in the High fuel-consumption band show the highest failure rate (42.28%). The pattern indicates fuel intensity may act as a practical indicator of operating stress and maintenance risk.

In [30]:
query = """
with total_fuel as (select fuel_class, count(*) as no_total_fuel
from failure_fe
group by fuel_class),
failed_fuel as (select fuel_class as failed_fuel_class, count(*) as failed_fuel
from failure_fe
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_e = 1 or failure_d = 1 or failure_e = 1
group by fuel_class)
select *, round(100.0 * failed_fuel / no_total_fuel, 2) as fuel_pct
from failed_fuel as ff
left join total_fuel as tf on ff.failed_fuel_class = tf.fuel_class 
order by fuel_pct desc

"""

con.execute(query).df()

,failed_fuel_class,failed_fuel,fuel_class,no_total_fuel,fuel_pct
0,High,2986,High,7063,42.28
1,Medium,615,Medium,2926,21.02
2,Low,13,Low,92,14.13


#### Temperature Insight

Failure likelihood increases across temperature bands from Low to High, indicating thermal load is directionally associated with risk. Because high-temperature groups can be smaller, this pattern should be validated in model-based analysis with uncertainty checks.

In [35]:
query = """
with total_temp as (select temp_class as total_temp_class, count(*) as total_no_temp
from failure_fe
group by total_temp_class),
failed_temp as (select temp_class, count(*) as no_failed_temp
from failure_fe
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1 
group by temp_class)
select *, round(100.0 * no_failed_temp / total_no_temp) as temp_pct
from failed_temp as ft
left join total_temp as tt on ft.temp_class = tt.total_temp_class
order by temp_pct desc

"""

con.execute(query).df()

,temp_class,no_failed_temp,total_temp_class,total_no_temp,temp_pct
0,High,43,High,64,67.0
1,Medium,2602,Medium,5876,44.0
2,Low,969,Low,4141,23.0


#### Failure-Type Prevalence

Failure D is the most common type while Failure C is least common. This distribution helps prioritize diagnostic attention toward the highest-frequency failure modes and informs maintenance triage.

In [48]:
query = """
select sum(case when failure_a = 1 then 1 else 0 end) as no_failure_a, sum(case when failure_b = 1 then 1 else 0 end) as no_failure_b, 
sum(case when failure_c = 1 then 1 else 0 end) as no_failure_c, sum(case when failure_d = 1 then 1 else 0 end) as no_failure_d, sum(case when failure_e = 1 then 1 else 0 end) as no_failure_e,
from failure_fe
"""

con.execute(query).df()

,no_failure_a,no_failure_b,no_failure_c,no_failure_d,no_failure_e
0,861.0,1044.0,809.0,1094.0,835.0


#### Single vs Multiple Failure Behavior

This analysis distinguishes isolated failures from co-occurring failures. A higher share of multiple failures can indicate systemic stress patterns rather than one-off component issues.

In [71]:
query = """
select case when failure_a + failure_b + failure_c + failure_d + failure_e = 0 then 'No Failure'
when failure_a + failure_b + failure_c + failure_d + failure_e = 1 then 'Single Failure' else 'Multiple Failures' end as Failure_group, count(*) as no_failure_group, round(100.0 * count(*) / sum(count (*)) over(), 2) as pct
from failure_fe
group by failure_group
order by pct desc
"""

con.execute(query).df()

,Failure_group,no_failure_group,pct
0,No Failure,6467,64.15
1,Single Failure,2738,27.16
2,Multiple Failures,876,8.69


#### Final Summary and Interpretation

#### Key Findings
- The Fail or Pass split (35.85% vs 64.15%) is sufficiently balanced to support meaningful baseline classification without severe minority-class sparsity.
- Model-level differences, led by SUV at 38.24% failure, indicate segment-specific risk behavior beyond random variation.
- Factory B’s higher failure rate suggests production-source effects that merit quality-control follow-up.
- The strong gradient by usage, including 71.36% in very high usage, points to operational intensity as a primary risk driver.
- Elevated risk in high fuel-consumption vehicles (42.28%) reinforces fuel intensity as a useful stress proxy.

#### Business Implications
- Maintenance scheduling should prioritize high-usage and high fuel-consumption cohorts where expected risk concentration is greatest.
- Factory and model segmentation can improve inspection allocation and root-cause investigation.
- Engineered operating-condition bands provide interpretable inputs for operational dashboards and model features.

#### Limitations
- Results are associative and should not be interpreted as causal effects.
- Potential confounding and feature interactions are not yet modeled.
- Some subgroup percentages may be sensitive to sample-size imbalance.
- The current notebook stops at EDA and target design, without predictive performance validation.

#### Recommended Next Steps
- Train baseline classifiers on the engineered target and compare model families.
- Evaluate performance using ROC-AUC, precision, recall, F1, and calibration checks.
- Apply cross-validation and decision-threshold tuning for maintenance use cases.
- Add explainability analysis to verify whether learned drivers align with domain expectations.
- Refactor notebook logic into reusable pipeline components for production readiness.

#### Project Summary
This analysis demonstrates a full workflow from raw-data quality treatment through interpretable risk segmentation and target engineering, providing a clear foundation for predictive maintenance modeling.